# Coerção de tipos

Quando o Pydantic desserializa dados, uma das coisas que ele faz é validar. Isso inclui garantir que os dados da instância terminem
com o tipo correto.

Vejamos um exemplo:

In [1]:
from pydantic import BaseModel, ValidationError

In [2]:
class Coordinates(BaseModel):
    x: float
    y: float

Repare que estamos exigindo que `x` e `y` sejam floats.

Vamos desserializar alguns dados:

In [3]:
p1 = Coordinates(x=1.1, y=2.2)
print(f"p1: {p1!r}")

p1: Coordinates(x=1.1, y=2.2)


Podemos ver as definições dos campos:

In [4]:
print(f"Coordinates.model_fields: {Coordinates.model_fields!r}")

Coordinates.model_fields: {'x': FieldInfo(annotation=float, required=True), 'y': FieldInfo(annotation=float, required=True)}


O tipo definido de cada campo é `float`.

E, de fato, se checarmos o tipo dos campos, temos o esperado:

In [5]:
print(f"type(p1.x): {type(p1.x)!r}")

type(p1.x): <class 'float'>


Mas o que acontece se os dados fornecidos não tiverem exatamente o tipo esperado?

O Pydantic tenta "transformar" os dados para o tipo correto — isso se chama **coerção** de tipos.

Vejamos:

In [6]:
p2 = Coordinates(x=0, y="1.2")
print(f"p2: {p2!r}")

p2: Coordinates(x=0.0, y=1.2)


O Pydantic conseguiu converter o **inteiro** `0` e a **string** `"1.2"` para float:

In [7]:
print(f"type(p2.x), type(p2.y): {type(p2.x), type(p2.y)!r}")

type(p2.x), type(p2.y): (<class 'float'>, <class 'float'>)


Nem sempre o Pydantic consegue fazer a coerção. Na verdade, podemos até escolher o nível de coerção aceitável.

Por padrão, a coerção é chamada de **lax** (frouxa) e tenta várias conversões.

Mas, como veremos depois, dá para mudar para o modo **strict** (estrito), bem menos tolerante com tipos incorretos nos dados.

A documentação do Pydantic que descreve quais coerções são tentadas em cada modo está aqui:

[https://docs.pydantic.dev/latest/concepts/conversion_table/](https://docs.pydantic.dev/latest/concepts/conversion_table/)

Olhando a tabela, você vai notar, por exemplo, que no modo lax, com tipos Python, entradas `float`, `int` ou `Decimal` viram float.
Já strings só viram float sob certas condições.

No modo strict, a conversão de string para float não é suportada (levanta erro de validação).

Consulte essa tabela ao pensar em coerção, porque as regras nem sempre são "óbvias".

Por exemplo, com este modelo:

In [8]:
class Model(BaseModel):
    field: str

Sabemos que todo objeto Python tem representação `str()`, então poderíamos esperar que qualquer tipo passado em `field` fosse
convertido para string.

Mas não é o caso:

In [9]:
try:
    Model(field=100)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field
  Input should be a valid string [type=string_type, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


E isso provavelmente é bom: permitir essa conversão poderia causar problemas, transformando em string um objeto que nunca
deveria virar string (a fonte de dados pode ter mudado — e a conversão automática esconderia um bug).

Veja o que quero dizer:

Estamos consultando uma API REST e recebendo um JSON, que modelamos assim:

In [10]:
class Contact(BaseModel):
    email: str

In [11]:
initial_json_data = '''
{
    "email": "inewton@principia.com"
}
'''

Isso desserializa sem problemas:

In [12]:
print(f"Contact.model_validate_json(initial_json_data): {Contact.model_validate_json(initial_json_data)!r}")

Contact.model_validate_json(initial_json_data): Contact(email='inewton@principia.com')


Agora suponha que a API mude o modelo de resposta, e não ficamos sabendo.

A resposta passa a ser assim:

In [13]:
new_json_data = '''
{
    "email": {
        "personal": "inewton@principia.com",
        "work": "isaac.newton@themint.com"
    }
}
'''

Tentar desserializar isso falha, e percebemos na hora que algo está errado na aplicação (e podemos corrigir):

In [14]:
try:
    Contact.model_validate_json(new_json_data)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Contact
email
  Input should be a valid string [type=string_type, input_value={'personal': 'inewton@pri...aac.newton@themint.com'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


E se o Pydantic tivesse simplesmente transformado esse objeto `email` (basicamente um dicionário aninhado) na sua representação em string?

Podemos imitar esse comportamento assim:

In [15]:
new_data = {
    "email": {
        "personal": "inewton@principia.com",
        "work": "isaac.newton@themint.com"
    }
}

In [16]:
print(f"Contact(email=str(new_data['email'])): {Contact(email=str(new_data['email']))!r}")

Contact(email=str(new_data['email'])): Contact(email="{'personal': 'inewton@principia.com', 'work': 'isaac.newton@themint.com'}")


Agora temos uma representação em string do dicionário de e-mails — nenhuma exceção, e o código provavelmente vai quebrar dali em
diante, dependendo de como usamos o campo `email`.